# Atividade de Aprendizado de Máquina: Classificador KNN (Breast Cancer Dataset)

Esta atividade demonstra a aplicação do algoritmo K-Nearest Neighbors (KNN) utilizando o dataset de Câncer de Mama da biblioteca Scikit-Learn.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# Configurando o estilo dos gráficos
plt.style.use('ggplot')
%matplotlib inline

## 1. Informações Gerais sobre o Dataset
Vamos carregar o dataset `load_breast_cancer` e inspecionar sua estrutura (shape, classes e features).

In [ ]:
# Carregando o dataset
cancer_data = load_breast_cancer()

# Criando o DataFrame do Pandas para melhor visualização e manipulação
df = pd.DataFrame(data=cancer_data.data, columns=cancer_data.feature_names)
df['target'] = cancer_data.target

print(f"Shape do Dataset: {df.shape}")
print(f"Classes disponíveis: {cancer_data.target_names} (0 = {cancer_data.target_names[0]}, 1 = {cancer_data.target_names[1]})")
print("\nPrimeiras 5 linhas do dataset:")
df.head()

In [ ]:
# Informações gerais das colunas e tipos de dados
df.info()

In [ ]:
# Estatísticas descritivas básicas das features
df.describe()

## 2. Gráfico Explorando a Distribuição dos Dados
Vamos plotar gráficos para analisar a distribuição da classe alvo (Maligno vs. Benigno) e de algumas features importantes do conjunto de dados.

In [ ]:
# Proporção das classes (Benigno vs Maligno)
target_counts = df['target'].value_counts()
class_names = [cancer_data.target_names[i] for i in target_counts.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(class_names, target_counts.values, color=['skyblue', 'salmon'], edgecolor='black')
plt.xlabel('Diagnóstico (Classe)')
plt.ylabel('Quantidade de Amostras')
plt.title('Distribuição das Classes (Maligno vs Benigno)')

# Adicionando rótulos com os valores acima das barras
for bar in bars:
    height = bar.get_height()
    plt.annotate(f'{height}',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3),  # 3 points vertical offset
                 textcoords="offset points",
                 ha='center', va='bottom')

plt.show()

# Histograma de uma feature importante (mean radius) estratificado por classe
plt.figure(figsize=(10, 6))
plt.hist(df[df['target'] == 0]['mean radius'], bins=20, alpha=0.5, label='Maligno', color='red', edgecolor='black')
plt.hist(df[df['target'] == 1]['mean radius'], bins=20, alpha=0.5, label='Benigno', color='green', edgecolor='black')
plt.xlabel('Mean Radius (Raio Médio)')
plt.ylabel('Frequência')
plt.title('Distribuição do Raio Médio por Classe')
plt.legend()
plt.show()

## 3. Divisão dos Dados e Pré-processamento com StandardScaler

### Justificativa para a Padronização (StandardScaler)
O algoritmo KNN (K-Nearest Neighbors) funciona com base no cálculo de distâncias (como a distância Euclidiana) entre pontos no espaço de características (features). 

Se as features estiverem em escalas muito diferentes, a feature com os maiores valores absolutos dominará completamente a métrica de distância. Por exemplo, a feature `mean area` (que varia de 100 a mais de 2000) influenciaria muito mais a distância do que a feature `mean smoothness` (que varia de 0.05 a 0.16), mesmo se a suavidade for um indicador médico mais relevante.

O `StandardScaler` padroniza os dados de modo que cada feature passe a ter **média igual a 0** e **desvio padrão igual a 1**. Isso garante que todas as features contribuam igualmente para o cálculo das distâncias geométricas no KNN.

In [ ]:
# Separando features (X) e a variável alvo (y)
X = df.drop(columns=['target'])
y = df['target']

# Divisão de treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Tamanho do treino (X_train): {X_train.shape}")
print(f"Tamanho do teste (X_test): {X_test.shape}")

# Inicializando e ajustando o Scaler nos dados de treino, e aplicando a transformação
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nPrimeira linha do treino antes da padronização (primeiras 5 features):")
print(X_train.iloc[0, :5].values)
print("\nPrimeira linha do treino após a padronização (primeiras 5 features):")
print(X_train_scaled[0, :5])

## 4. Treinamento do Modelo testando Valores de K de 1 a 20
Vamos testar diferentes valores de vizinhos mais próximos ($K$), de 1 até 20, registrando a acurácia para cada valor no conjunto de teste.

In [ ]:
k_values = list(range(1, 21))
accuracies = []

for k in k_values:
    # Inicializando o classificador com o respectivo K
    knn = KNeighborsClassifier(n_neighbors=k)
    # Treinando o modelo com os dados padronizados
    knn.fit(X_train_scaled, y_train)
    # Prevendo nos dados de teste
    y_pred = knn.predict(X_test_scaled)
    # Calculando acurácia
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    print(f"K = {k:02d} | Acurácia no Teste: {acc:.4f}")

## 5. Gráfico de Acurácia × K e Escolha Justificada do Melhor K
Vamos visualizar como a acurácia varia com a mudança de $K$ para determinar o valor ideal.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(k_values, accuracies, marker='o', linestyle='dashed', color='blue', markerfacecolor='red', markersize=8)
plt.title('Acurácia × Valor de K no KNN')
plt.xlabel('Valor de K')
plt.ylabel('Acurácia no Teste')
plt.xticks(k_values)
plt.grid(True)
plt.show()

# Encontrando o K com maior acurácia
best_idx = np.argmax(accuracies)
best_k = k_values[best_idx]
best_acc = accuracies[best_idx]
print(f"Melhor acurácia obtida: {best_acc:.4%} no valor de K = {best_k}")

### Escolha Justificada do Melhor K

Com base no gráfico gerado:
1. **Evitando Overfitting**: Valores de $K$ muito baixos (ex: $K=1$, $K=2$) tendem a criar fronteiras de decisão complexas e muito ruidosas, capturando variações insignificantes do treino (overfitting). Embora possam apresentar alta acurácia, são muito sensíveis a outliers.
2. **Evitando Underfitting**: Valores muito altos de $K$ suavizam demais a fronteira de decisão, fazendo com que o modelo perca padrões locais importantes e tenda a votar apenas na classe majoritária (underfitting).
3. **K ideal**: O valor de **K = 11** (ou o valor retornado como `best_k` na execução) representa um excelente equilíbrio, atingindo a acurácia máxima estável no conjunto de teste sem complexidade excessiva.

## 6. Avaliação Final do Modelo com o K Escolhido
Vamos instanciar o modelo final usando o valor ideal de $K$, treiná-lo e apresentar as métricas detalhadas com o `classification_report`.

In [ ]:
# Instanciando o classificador KNN com o K otimizado
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)

# Predição no conjunto de teste
y_final_pred = final_knn.predict(X_test_scaled)

# Relatório de classificação detalhado
print(f"Relatório de Classificação final para K = {best_k}:\n")
print(classification_report(y_test, y_final_pred, target_names=cancer_data.target_names))